<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
# 2 · Word Vectors and Neural Networks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/02-word-vectors.ipynb)

**From Traditional NLP to Modern LLMs** · Day 1 · 95 minutes (45 briefing, 50 lab)

From one-hot vectors to learned embeddings, with a PyTorch refresher and the first neural language model.

**By the end of this notebook you can:**

- Explain the distributional hypothesis and find nearest neighbors by cosine similarity
- Derive the skip-gram negative-sampling loss, implement it, and check its gradients against autograd
- Assemble a feed-forward classifier over averaged embeddings and evaluate it against TF-IDF

**Stack:** PyTorch

In [ ]:
#@title Workshop harness: run this cell first { display-mode: "form" }
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Tick WORKED_EXAMPLE to run every reference solution in place of your code. That gives a
# complete worked example; it does not show that you did the exercises.
WORKED_EXAMPLE = False  #@param {type:"boolean"}

import os as _os
import time as _time

from IPython import get_ipython as _get_ipython

_NOTEBOOK = "02-word-vectors"
_CONTENT_SHA = "1a263cb476488a94"
_EXERCISES = {1: ('sgns_loss',), 2: ('nearest_neighbors',), 3: ('average_embeddings',), 4: ('FeedForwardClassifier',)}
_HINTS = {1, 2, 3}  # exercises with a folded Hint cell above their TODO cell
_PATHS = {'offline': 'offline stand-ins and test doubles: checks that the code runs, not what a model does', 'open': 'real open models from the Hugging Face Hub, no API keys', 'keyed': 'commercial APIs with keys'}
_FALLBACK_NOTE = None
_RECORDED_SETTINGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE', 'NLP_LLMS_QUICK', 'HF_HUB_OFFLINE')
_OFFLINE_FLAGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE')
_FALLBACK_MARKERS = ('Could not load ', 'did not load', 'USING StubProvider', 'LSA stand-in fitted', 'OFFLINE TEST MODE')
_RECORD_PACKAGES = (
    "torch", "transformers", "numpy", "scikit-learn", "peft", "datasets", "openai",
    "anthropic", "typesafe-sdk", "langgraph", "langchain-core", "llama-index-core",
)


class _Workshop:
    """Keeps the folded solutions from replacing your code, and says whose code each
    checkpoint checked: yours, the reference solution, or code the lab provides."""

    _MISSING = object()

    def __init__(self, old=None):
        # Rerunning this cell (as Run all does) keeps the stored solutions but starts a new
        # run: the checkpoint results, cell times and run clock describe this run only.
        self.ref = getattr(old, "ref", {})
        self.stub = getattr(old, "stub", {})
        self.chosen = getattr(old, "chosen", set())  # exercises switched by use_reference
        self.verified = {}
        self._verifying = None
        self.results = {}
        self.cell_seconds = []
        self.cell_errors = 0
        self.fallbacks = set()
        self.started = _time.time()
        self._current = None
        self._t0 = None

    @property
    def worked(self):
        if _os.environ.get("NLP_LLMS_WORKED") == "1":
            return True
        return bool(_get_ipython().user_ns.get("WORKED_EXAMPLE"))

    def _is_reference(self, n, name, obj):
        return any(obj is r for r in self.ref.get(n, {}).get(name, ()))

    def solution(self, n):
        """Decorator on a reference solution: stores it and, unless WORKED_EXAMPLE is
        ticked, leaves your own definition of the same name in place."""

        def bind(obj, name=None):
            name = name or obj.__name__
            self.ref.setdefault(n, {}).setdefault(name, []).append(obj)
            current = _get_ipython().user_ns.get(name, self._MISSING)
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.stub.setdefault(n, {})[name] = current
            if self.worked:
                return obj
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.chosen.discard(n)  # your TODO cell ran again: back to your code
                return current
            if n in self.chosen:
                return obj  # you chose the reference with use_reference(n)
            # Your TODO cell has not run, or the name still holds a reference from a
            # worked run: go back to your code if it is known, never silently to the reference.
            yours = self.stub.get(n, {}).get(name, self._MISSING)
            if yours is not self._MISSING:
                return yours
            if callable(obj):
                return self._not_run(n, name)
            print(f"[workshop] Run your # TODO {n} cell: {name} is the reference until you do.")
            return obj

        return bind

    def _not_run(self, n, name):
        """Stands in for a function or class whose TODO cell has not run yet."""

        def missing(*args, **kwargs):
            raise NotImplementedError(
                f"{name}: run your # TODO {n} cell first, or go on with"
                f" workshop.use_reference({n!r})"
            )

        missing.__name__ = name
        return missing

    def solution_value(self, n, name, value):
        """The same as solution(n), for a value rather than a function or class."""
        if value is None or isinstance(value, (bool, int, float, str, bytes)):
            # Python shares equal small values, so yours and the reference could not be told
            # apart: give such a value in the stub instead of making it part of the exercise.
            kind = type(value).__name__
            raise TypeError(f"solution_value({n!r}, {name!r}): use an object, not a plain {kind}")
        return self.solution(n)(value, name)

    def use_reference(self, n):
        """Go on with the reference solution of exercise n in place of yours."""
        if n not in self.ref:
            raise KeyError(f"Exercise {n} has no reference solution yet: run its Solution cell")
        ns = _get_ipython().user_ns
        for name, objs in self.ref[n].items():
            ns[name] = objs[-1]
        self.chosen.add(n)
        print(
            f"Exercise {n}: now using the REFERENCE solution. Rerun the cells below it."
            f" To switch back to your code, rerun your # TODO {n} cell."
        )

    def checkpoint(self, n=None, label=None):
        """First line of a checkpoint cell: notes whose code the cell is about to check."""
        names = _EXERCISES.get(n, ())
        ns = _get_ipython().user_ns
        k = sum(self._is_reference(n, x, ns.get(x, self._MISSING)) for x in names)
        if not names:
            whose = None  # the results of the cells above: provided code, maybe your earlier work
        elif k == len(names):
            whose = "the REFERENCE solution"
        elif k == 0:
            whose = "your code"
        else:
            whose = "partly the REFERENCE solution"
        self._current = (n, str(label if label is not None else n), whose)

    def _pre(self, info):
        self._t0 = _time.time()

    def _post(self, result):
        if self._t0 is None:  # this harness cell itself: its hooks were registered mid-cell
            return
        self.cell_seconds.append(round(_time.time() - self._t0, 2))
        if not result.success and self._verifying is None:
            self.cell_errors += 1  # a verification copy fails on purpose; it is not an error
        if self._current is None:
            return
        n, label, whose = self._current
        self._current = None
        if self._verifying is not None:
            self.verified[self._verifying] = bool(result.success)
            return
        self.results[label] = (bool(result.success), whose)
        if whose is None:
            if result.success:
                print(f"[workshop] Checkpoint {label} passed.")
            elif isinstance(result.error_in_exec, NotImplementedError):
                print(
                    f"[workshop] Checkpoint {label} uses an exercise you have not written yet."
                    " Finish it, or go on with workshop.use_reference(N) for that exercise."
                )
            else:
                print(f"[workshop] Checkpoint {label} failed. Read the message above.")
        elif result.success:
            print(f"[workshop] Checkpoint {label} passed on {whose}.")
        elif isinstance(result.error_in_exec, NotImplementedError):
            print(
                f"[workshop] Checkpoint {label}: TODO {n} is not written yet. Write it, run"
                " its cell, then rerun this one. To go on without it:"
                f" workshop.use_reference({n!r})"
            )
            if n in _HINTS:
                print(f"[workshop] Stuck? Open the folded Hint above TODO {n} first.")
        elif whose == "the REFERENCE solution":
            print(
                f"[workshop] Checkpoint {label} failed on the reference solution: a problem"
                " with the lab or the runtime, not with your code. Tell the instructor."
            )
        else:
            print(
                f"[workshop] Checkpoint {label} failed on {whose}. Read the assertion above,"
                f" fix TODO {n}, or go on with workshop.use_reference({n!r})."
            )
            if n in _HINTS:
                print(f"[workshop] The folded Hint above TODO {n} says what to check.")

    def _verify_begin(self, n):
        """Test-only (scripts/test_notebooks.py --verify-checkpoints): bind exercise n's
        stubs, so that a copy of its checkpoint runs on unfinished code."""
        ns = _get_ipython().user_ns
        self._saved = {name: ns.get(name, self._MISSING) for name in _EXERCISES.get(n, ())}
        ns.update(self.stub.get(n, {}))
        self._verifying = n

    def _verify_end(self, n):
        """Test-only: restore the reference and require that the stub copy failed."""
        ns = _get_ipython().user_ns
        for name, value in self._saved.items():
            if value is self._MISSING:
                ns.pop(name, None)
            else:
                ns[name] = value
        self._verifying = None
        if self.verified.get(n) is not False:
            raise AssertionError(
                f"The checkpoint of exercise {n} passed on the unfinished stub: it cannot tell"
                " a finished exercise from an unfinished one"
            )

    def summary(self):
        """Which checkpoints passed, and on whose code."""
        groups = {"your code": [], "the reference": [], "other checks": [], "failed": []}
        for label, (ok, whose) in self.results.items():
            if not ok:
                groups["failed"].append(label)
            elif whose is None:
                groups["other checks"].append(label)
            elif whose == "your code":
                groups["your code"].append(label)
            else:
                groups["the reference"].append(label)
        mode = "WORKED EXAMPLE: reference solutions" if self.worked else "your code"
        print(f"{_NOTEBOOK}, run with {mode}.")
        path = self.path()
        # _PATHS comes from readiness.paths in _variables.yml; the offline entry says that
        # such numbers check the code, not a model.
        print(f"  What ran: {_PATHS[path]}.")
        if path == "open" and _FALLBACK_NOTE:
            print(f"  Read the numbers with this in mind: {_FALLBACK_NOTE}.")
        lines = {
            "your code": "Checkpoints passed on your code",
            "the reference": "Checkpoints passed on a reference solution",
            "other checks": "Other checks passed",
            "failed": "Checkpoints failed",
        }
        for group, labels in groups.items():
            print(f"  {lines[group]}: {', '.join(labels) or 'none'}")
        if self.worked:
            print("  A worked-example run shows how the lab goes, not that you did it.")

    def path(self):
        """offline, keyed or open: the same rule as scripts/add_run_record.py's path_of."""
        ns = _get_ipython().user_ns
        if self.fallbacks or any(_os.environ.get(flag) for flag in _OFFLINE_FLAGS):
            return "offline"
        if ns.get("PROVIDER") in ("openai", "anthropic") or ns.get("JEV_PATH") == "keyed":
            return "keyed"
        return "open"

    def run_record(self):
        """A run record for runs/ (see runs/README.md): paste it into
        scripts/add_run_record.py. It holds no code, outputs, keys or names."""
        import importlib.metadata as _md
        import json
        import platform
        import sys

        gpu = None
        try:
            import torch

            if torch.cuda.is_available():
                gpu = torch.cuda.get_device_name(0)
        except Exception:
            pass
        packages = {}
        for name in _RECORD_PACKAGES:
            try:
                packages[name] = _md.version(name)
            except Exception:
                pass
        import datetime as _dt

        record = {
            "date": _dt.datetime.now(_dt.timezone.utc).date().isoformat(),
            "notebook": _NOTEBOOK,
            "content_sha": _CONTENT_SHA,
            "mode": "worked" if self.worked else "learner",
            # A run passes when it reached this last cell with no error on the way and every
            # checkpoint passed. Run all reruns the harness first, so it starts clean.
            "status": "pass"
            if self.results and self.cell_errors == 0 and all(ok for ok, _ in self.results.values())
            else "fail",
            "cell_errors": self.cell_errors,
            "fallbacks": sorted(self.fallbacks),
            "seconds": round(_time.time() - self.started, 1),
            "cell_seconds": self.cell_seconds,
            "python": platform.python_version(),
            "platform": sys.platform,
            "gpu": gpu,
            "colab_release": _os.environ.get("COLAB_RELEASE_TAG"),
            "provider": _get_ipython().user_ns.get("PROVIDER"),
            "jev_path": _get_ipython().user_ns.get("JEV_PATH"),
            "settings": {k: _os.environ[k] for k in _RECORDED_SETTINGS if _os.environ.get(k)},
            "packages": packages,
        }
        print("----- run record (paste into scripts/add_run_record.py) -----")
        print(json.dumps(record))
        print("----- end of run record -----")


_ip = _get_ipython()
_old = _ip.user_ns.get("workshop")
# Remove only an earlier harness's own hooks; every other hook (Colab's, an extension's)
# stays registered.
for _event in ("pre_run_cell", "post_run_cell"):
    for _callback in list(_ip.events.callbacks[_event]):
        if type(getattr(_callback, "__self__", None)).__name__ == "_Workshop":
            _ip.events.unregister(_event, _callback)
workshop = _Workshop(_old if hasattr(_old, "ref") else None)

# Notice when a lab falls back to a stand-in, for the run record. print is wrapped once.
import builtins as _builtins

if not getattr(_builtins.print, "_workshop", False):
    _print = _builtins.print

    def _watching_print(*args, **kwargs):
        try:
            ip = _get_ipython()
            ws = ip.user_ns.get("workshop") if ip is not None else None
            if ws is not None and hasattr(ws, "fallbacks"):
                # Labs print their fallback notices as strings; never stringify other objects.
                text = " ".join(a for a in args if isinstance(a, str))
                ws.fallbacks.update(m.strip() for m in _FALLBACK_MARKERS if m in text)
        except Exception:
            pass  # watching must never break printing
        return _print(*args, **kwargs)

    _watching_print._workshop = True
    _builtins.print = _watching_print
_ip.events.register("pre_run_cell", workshop._pre)
_ip.events.register("post_run_cell", workshop._post)
print("Workshop harness ready:", "WORKED EXAMPLE" if workshop.worked else "checking your code")


## Setup

**Runtime.** Use a GPU runtime if you can: *Runtime → Change runtime type → T4 GPU*. The lab also runs on a CPU runtime; only the embedding training (Section A2) is slower there. On 2 CPU threads of a heavily shared machine it took 8 to 12 minutes, and Run all 11 to 16; it has not yet been timed on Colab, on either a CPU or a T4 runtime. No API key and no installs are needed: PyTorch, NumPy, scikit-learn and matplotlib are preinstalled on Colab.

**How the lab works.** There are four exercises and one optional stretch section. Each exercise has:

1. a **Predict** question to answer before you run anything;
2. a `# TODO` cell where you write one short function;
3. a folded **Solution** cell (open it only after trying);
4. a **Checkpoint** cell that tells you whether your function is right.

The folded solution cells never replace your code: each checkpoint checks your own functions and says so. If you are stuck, `workshop.use_reference(N)` lets you go on with exercise N's reference solution, and the checkpoint then says it checked the reference. To run the whole lab as a worked example, tick `WORKED_EXAMPLE` in the harness cell at the top; a worked run shows how the lab goes, not that you did it.

**What you build.** Part A trains word embeddings with skip-gram and negative sampling (SGNS) and inspects them. Part B uses them as features for the Lab 1 classification task and puts the result next to Lab 1's TF-IDF baseline, on the same test split.

In [ ]:
# Setup: imports, seeds and device. Everything here is preinstalled on Colab.
import csv
import json
import math
import random
import re
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NAVY, ACCENT = "#16324f", "#b3541e"  # plot colors
GROUP_COLORS = [NAVY, ACCENT, "#2f7d4a", "#7a4fa3"]
LAB_START = time.time()
print(f"PyTorch {torch.__version__} · NumPy {np.__version__} · scikit-learn {sklearn.__version__}")
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else
                             " (no GPU: Section A2 will take a few minutes)"))

## Data

This lab uses the classification set of the running thread, the same file and the same split as Lab 1: titles and abstracts of arXiv papers in 4 classes, 4,800 / 600 / 1,600 papers for train / validation / test.

It is used twice:

- **Embedding corpus (Part A).** The text of the **training split only**, labels ignored. It is in-domain for the classifier, needs no second download, and keeps the test split unseen.
- **Classification (Part B).** The same task as Lab 1, scored with the same metrics (accuracy and macro-F1 on the test split).

The next cell is the workshop's standard loading cell, the same in every lab (see `data/README.md` in the repository).

In [ ]:
import csv, gzip, hashlib, io, os, urllib.request
from pathlib import Path


def fetch(name, urls, sha256):
    """Return the bytes of a workshop data file, checked against its SHA-256."""
    cache = Path(os.environ.get("NLP_LLMS_DATA", "data"))
    for path in (cache / name, Path("../data") / name):
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == sha256:
            return path.read_bytes()
    for url in urls:  # canonical URL first, fallback second
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                blob = response.read()
        except OSError as error:
            print(f"Could not fetch {url}: {error}")
            continue
        if hashlib.sha256(blob).hexdigest() != sha256:
            print(f"Ignoring {url}: the file does not match the expected hash")
            continue
        cache.mkdir(parents=True, exist_ok=True)
        (cache / name).write_bytes(blob)
        return blob
    raise RuntimeError(f"Could not load {name} from any source")


TOPIC_CLASSES = ["cs.CL", "cs.CV", "cs.LG", "cs.RO"]  # label 0, 1, 2, 3
TOPIC_NAMES = ["Language", "Vision", "Machine learning", "Robotics"]


def load_topics():
    """arXiv Topics v1. Returns {"train" | "val" | "test": (texts, labels)}."""
    blob = fetch(
        "arxiv_topics_v1.csv.gz",
        [
            "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/arxiv_topics_v1.csv.gz",
            "https://cdn.jsdelivr.net/gh/project-delphi/nlp-llms@main/data/arxiv_topics_v1.csv.gz",
        ],
        "49b51502fc6cf6ecff23fa266c5372594a88414dd96513579237792710246dc5",
    )
    splits = {"train": ([], []), "val": ([], []), "test": ([], [])}
    for row in csv.DictReader(io.StringIO(gzip.decompress(blob).decode("utf-8"))):
        texts, labels = splits[row["split"]]
        texts.append(row["title"] + "\n" + row["abstract"])
        labels.append(int(row["label"]))
    return splits


def load_lm_corpus():
    """Tiny Shakespeare. Returns {"train" | "val" | "test": str}, split by character offset."""
    blob = fetch(
        "tinyshakespeare.txt",
        [
            "https://raw.githubusercontent.com/karpathy/char-rnn/6f9487a6fe5b420b7ca9afb0d7c078e37c1d1b4e/data/tinyshakespeare/input.txt",
            "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/tinyshakespeare.txt",
        ],
        "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed",
    )
    text = blob.decode("utf-8")
    return {"train": text[:1_000_000], "val": text[1_000_000:1_055_000], "test": text[1_055_000:]}

**Tokenizer and vocabulary: Lab 1's rules, unchanged.** The tokenizer is the Lab 1 Exercise 1 function. The vocabulary is the one Lab 1's classifier used: every token that occurs in at least 2 training documents (`min_df=2`). Using the same vocabulary for TF-IDF and for the embeddings means the comparison in Part B changes the features and nothing else.

Here each word gets an integer ID in order of frequency. ID 0 is reserved for padding (`<pad>`), which Part B needs. Tokens outside the vocabulary are dropped, as word2vec drops words below its minimum count.

In [ ]:
topics = load_topics()  # {"train" | "val" | "test": (texts, labels)}
train_texts, train_labels = topics["train"]
val_texts, val_labels = topics["val"]
test_texts, test_labels = topics["test"]
y_train, y_val, y_test = np.array(train_labels), np.array(val_labels), np.array(test_labels)


def tokenize(text):
    # Lab 1, Exercise 1: lowercase; runs of letters, digits or apostrophes, or one other non-space symbol.
    return re.findall(r"[a-z0-9']+|[^a-z0-9'\s]", text.lower())


tokens = {split: [tokenize(t) for t in texts] for split, (texts, _) in topics.items()}
token_counts = Counter(t for doc in tokens["train"] for t in doc)

# Lab 1's classifier vocabulary: tokens in at least 2 training documents.
lab1_vocabulary = CountVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, min_df=2)
lab1_vocabulary.fit(train_texts)
WORDS = ["<pad>"] + sorted(lab1_vocabulary.vocabulary_, key=lambda w: (-token_counts[w], w))
WORD_TO_ID = {w: i for i, w in enumerate(WORDS)}
VOCAB_SIZE = len(WORDS)  # |V|, including <pad>

# Each document as an array of word IDs; out-of-vocabulary tokens are dropped.
doc_ids = {
    split: [np.array([WORD_TO_ID[t] for t in doc if t in WORD_TO_ID], dtype=np.int64) for doc in docs]
    for split, docs in tokens.items()
}

n_train_tokens = sum(len(doc) for doc in tokens["train"])
kept = sum(len(ids) for ids in doc_ids["train"])
print(f"Training split: {len(train_texts):,} papers, {n_train_tokens:,} tokens, "
      f"{len(token_counts):,} distinct tokens")
print(f"Vocabulary (min_df = 2): {VOCAB_SIZE - 1:,} words + <pad>; covers {kept / n_train_tokens:.1%} "
      "of training tokens")
for split in ("val", "test"):
    n = sum(len(doc) for doc in tokens[split])
    print(f"  coverage of {split:<4} tokens: {sum(len(ids) for ids in doc_ids[split]) / n:.1%}")
print("Most frequent:", WORDS[1:16])

# Part A · Word Vectors With Skip-Gram and Negative Sampling

## A1 · Training Pairs and the Noise Distribution (provided)

Skip-gram turns the corpus into (center, context) pairs $(w, o)$: each word $w_t$ is paired with the words $w_{t+j}$ in a window of half-width $m$ around it (briefing, Section 3). Three details of the provided pipeline, all from word2vec (Mikolov et al., 2013):

- **Windows stay inside one paper.** Pairs never cross from one abstract into the next.
- **Dynamic window.** For each center word a width $b$ is drawn uniformly from $1, \dots, m$, and only context words within $b$ positions are used. Near neighbors are used more often than far ones.
- **Subsampling of frequent words** (the optional box in the briefing). Before pairing, each occurrence of word $v$ is dropped with probability $\max\bigl(0,\ 1 - \sqrt{\tau / f(v)}\bigr)$, where $f(v)$ is its relative frequency. This thins out tokens such as *the*, *,* and *of*, which say little about their neighbors. The pairs are redrawn every epoch.

The noise distribution for negative samples is @eq-noise of the briefing, the unigram distribution raised to the power $3/4$: `noise_distribution` below.

In [ ]:
D = 100  # embedding dimension d
WINDOW = 5  # window half-width m
K = 5  # negative samples per real pair
TAU = 1e-3  # subsampling threshold tau (word2vec's default for its own corpora is 1e-3 to 1e-5)

train_ids = np.concatenate(doc_ids["train"])  # (N_tokens,) the training text as one ID sequence
train_doc = np.concatenate([np.full(len(ids), i) for i, ids in enumerate(doc_ids["train"])])  # paper of each token
word_counts = np.bincount(train_ids, minlength=VOCAB_SIZE).astype(np.float64)  # c(v); c(<pad>) = 0


def noise_distribution(counts, power=0.75):
    # Briefing @eq-noise: P_n(v) = c(v)^(3/4) / sum_v' c(v')^(3/4).
    weights = counts**power
    return weights / weights.sum()


def make_pairs(ids, doc, counts, rng, window=WINDOW, tau=TAU):
    # Skip-gram (center, context) pairs with subsampling and a dynamic window. Returns two (P,) arrays.
    freq = counts[ids] / counts.sum()  # f(v) of each token
    keep = rng.random(len(ids)) >= 1 - np.sqrt(tau / freq)  # subsampling
    ids, doc = ids[keep], doc[keep]
    width = rng.integers(1, window + 1, size=len(ids))  # dynamic window b for each center
    centers, contexts = [], []
    for j in range(1, window + 1):
        same_doc = doc[:-j] == doc[j:]
        right = same_doc & (width[:-j] >= j)  # center at t, context at t + j
        centers.append(ids[:-j][right])
        contexts.append(ids[j:][right])
        left = same_doc & (width[j:] >= j)  # center at t + j, context at t
        centers.append(ids[j:][left])
        contexts.append(ids[:-j][left])
    return np.concatenate(centers), np.concatenate(contexts)


P_NOISE = noise_distribution(word_counts)
_centers, _contexts = make_pairs(train_ids, train_doc, word_counts, np.random.default_rng(SEED))
print(f"One epoch: {len(train_ids):,} tokens -> {len(_centers):,} (center, context) pairs after subsampling")
print("Three sample pairs:", [(WORDS[c], WORDS[o]) for c, o in zip(_centers[1000:1003], _contexts[1000:1003])])

print("\nword       raw frequency  noise probability P_n")
for w in ["the", ",", "model", "robot", "panoptic"]:
    print(f"{w:<10} {word_counts[WORD_TO_ID[w]] / word_counts.sum():>13.5f}  {P_NOISE[WORD_TO_ID[w]]:>21.5f}")

The last table shows what the exponent $3/4$ does: frequent words are drawn as negatives less often than their raw frequency, rare words more often.

## Exercise 1 · The SGNS Loss (12 minutes)

For one real pair $(w, o)$ and $K$ noise words $\tilde{w}_1, \dots, \tilde{w}_K$ drawn from $P_n$, the loss is the briefing's @eq-sgns:

$$
\mathcal{L}_{\text{SGNS}}(w, o) = -\log \sigma(u_o^\top e_w) \; - \sum_{k=1}^{K} \log \sigma(-u_{\tilde{w}_k}^\top e_w).
$$

$e_w$ is the center vector of $w$ (a row of $E$), $u_o$ the context vector of $o$ (a row of $U$), and $\sigma$ the sigmoid. Your function gets the vectors already looked up, for a batch of $B$ pairs, and returns the mean over the batch.

**Predict.** All vectors are zero. What is the loss for $K = 5$? (Every score is 0, and $\sigma(0) = 1/2$.)

**Task.** Complete `sgns_loss`. Use `F.logsigmoid(x)` for $\log \sigma(x)$: computing `torch.log(torch.sigmoid(x))` gives $-\infty$ once $\sigma(x)$ rounds to 0. Broadcasting does the $K$ dot products at once: `e_center.unsqueeze(1)` has shape `(B, 1, d)`.

<details>
<summary>Hint for TODO 1</summary>

Each noise word contributes $-\log \sigma(-u_{\tilde{w}}^\top e_w)$: the minus sign goes inside the sigmoid, because $1 - \sigma(x) = \sigma(-x)$ (briefing @eq-sgns). Sum the $K$ noise terms of each pair, then average over the $B$ pairs; averaging over $K$ as well shrinks the noise part. With all-zero vectors every term is $\log 2$, so the loss must be $(K + 1)\log 2 \approx 4.16$.

</details>

In [ ]:
# TODO 1: complete this function.
def sgns_loss(e_center, u_context, u_negatives):
    # Mean SGNS loss (briefing @eq-sgns) over a batch of B real pairs.
    #   e_center:    (B, d)     center vectors e_w
    #   u_context:   (B, d)     context vectors u_o of the real context words
    #   u_negatives: (B, K, d)  context vectors of the K noise words of each pair
    # Returns a scalar tensor.
    raise NotImplementedError("TODO 1")

In [ ]:
#@title Solution 1 — try it yourself first { display-mode: "form" }
@workshop.solution(1)
def sgns_loss(e_center, u_context, u_negatives):
    # Mean SGNS loss (briefing @eq-sgns) over a batch of B real pairs.
    pos_score = (e_center * u_context).sum(dim=-1)  # (B,)    u_o^T e_w
    neg_score = (u_negatives * e_center.unsqueeze(1)).sum(dim=-1)  # (B, K)  u_k^T e_w
    loss = -F.logsigmoid(pos_score) - F.logsigmoid(-neg_score).sum(dim=-1)  # (B,)
    return loss.mean()

<details>
<summary>Why this solution works</summary>

The two lines of scores are the dot products of @eq-binary: one per real pair, shape `(B,)`, and $K$ per pair for the noise words, shape `(B, K)`. Broadcasting `(B, K, d) * (B, 1, d)` multiplies each noise vector by its own pair's center vector; summing over the last axis gives the dot products. The loss line is @eq-sgns term by term, using $1 - \sigma(x) = \sigma(-x)$ for the noise words.

At zero vectors every score is 0 and every term is $-\log \tfrac12 = \log 2$, so the loss is $(K + 1)\log 2 \approx 4.16$ for $K = 5$. That is the loss of a model that cannot tell real pairs from noise, and the value the training curve below starts from.

`F.logsigmoid` computes $\log \sigma(x) = -\log(1 + e^{-x})$ in a form that stays finite for large $|x|$. Autograd then gives the gradients of @eq-sgns-grad, divided by $B$ because of the mean; the checkpoint compares them.

</details>

In [ ]:
# Checkpoint 1
workshop.checkpoint(1)
def _logsig(x):
    return -math.log1p(math.exp(-x))


# (a) All-zero inputs give exactly (K + 1) log 2.
_loss = sgns_loss(torch.zeros(3, 8), torch.zeros(3, 8), torch.zeros(3, K, 8))
assert _loss.dim() == 0, "return a scalar: the mean over the batch"
assert math.isclose(_loss.item(), (K + 1) * math.log(2), rel_tol=1e-6), (
    f"all-zero vectors: expected (K + 1) log 2 = {(K + 1) * math.log(2):.4f}, got {_loss.item()}: "
    "sum the K noise terms of each pair, then average over the batch")

# (b) A hand-computed batch of two pairs. Pair 1 scores: real 2, noise 1 and 0. Pair 2: real -1, noise 0.5 and 3.
_e = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
_u = torch.tensor([[2.0, 0.0], [0.0, -1.0]])
_un = torch.tensor([[[1.0, 0.0], [0.0, 3.0]], [[2.0, 0.5], [0.0, 3.0]]])
_expected = (-_logsig(2) - _logsig(-1) - _logsig(0) - _logsig(-1) - _logsig(-0.5) - _logsig(-3)) / 2
assert math.isclose(sgns_loss(_e, _u, _un).item(), _expected, rel_tol=1e-5), (
    f"hand-computed batch: expected {_expected:.5f}, got {sgns_loss(_e, _u, _un).item():.5f}: "
    "each noise term is log sigmoid(-u_k . e_w), with the minus sign inside the sigmoid")

# (c) Large scores stay finite (real-pair score -200, noise score +200: loss 400).
_big = sgns_loss(torch.full((1, 2), 10.0), torch.full((1, 2), -10.0), torch.full((1, 1, 2), 10.0))
assert torch.isfinite(_big), "use F.logsigmoid: log(sigmoid(x)) underflows to -inf"
assert math.isclose(_big.item(), 400.0, rel_tol=1e-4), (
    f"real-pair score -200, noise score +200: expected loss 400, got {_big.item()}: use F.logsigmoid, not a clamped log")

# (d) Autograd's gradients equal the briefing's @eq-sgns-grad (divided by B for the batch mean).
_g = torch.Generator().manual_seed(0)
_B = 6
_e = torch.randn(_B, 8, generator=_g, requires_grad=True)
_u = torch.randn(_B, 8, generator=_g, requires_grad=True)
_un = torch.randn(_B, K, 8, generator=_g, requires_grad=True)
sgns_loss(_e, _u, _un).backward()
with torch.no_grad():
    _s_pos = torch.sigmoid((_e * _u).sum(-1, keepdim=True))  # (B, 1)  sigma(u_o^T e_w)
    _s_neg = torch.sigmoid((_un * _e.unsqueeze(1)).sum(-1, keepdim=True))  # (B, K, 1)
    _grad_e = (-(1 - _s_pos) * _u + (_s_neg * _un).sum(1)) / _B
    _grad_u = -(1 - _s_pos) * _e / _B
    _grad_un = _s_neg * _e.unsqueeze(1) / _B
assert torch.allclose(_e.grad, _grad_e, atol=1e-6), "gradient with respect to e_w differs from @eq-sgns-grad"
assert torch.allclose(_u.grad, _grad_u, atol=1e-6), "gradient with respect to u_o differs"
assert torch.allclose(_un.grad, _grad_un, atol=1e-6), "gradient with respect to the noise vectors differs"
print("Checkpoint 1 passed: loss values and all three gradients match @eq-sgns and @eq-sgns-grad")

**Why not the full softmax?** The provided cell below times one training step of skip-gram with the full softmax (@eq-softmax and @eq-sg-loss of the briefing) against one step with your `sgns_loss`, on the same batch. You do not need to read the code; read the timings.

In [ ]:
# Provided: full-softmax skip-gram (briefing @eq-softmax, @eq-sg-loss), for a timing comparison only.
def softmax_skipgram_loss(e_center, context_ids, U_all):
    scores = e_center @ U_all.T  # (B, |V|): one score for every word in the vocabulary
    return F.cross_entropy(scores, context_ids)  # mean of -log P(o | w)


def time_steps(vocab_size, batch=256, repeats=5):
    # Seconds per forward + backward pass for (full softmax, SGNS) on random vectors.
    # Each loss gets exactly the vectors it reads: the softmax needs all |V| context vectors,
    # SGNS needs 1 + K of them per pair.
    def leaf(*shape):
        return torch.randn(*shape, device=DEVICE, requires_grad=True)

    e, U_all = leaf(batch, D), leaf(vocab_size, D)
    u, u_neg = leaf(batch, D), leaf(batch, K, D)
    context = torch.randint(0, vocab_size, (batch,), device=DEVICE)
    steps = {
        "softmax": lambda: softmax_skipgram_loss(e, context, U_all).backward(),
        "sgns": lambda: sgns_loss(e, u, u_neg).backward(),
    }
    seconds = {}
    for name, step in steps.items():
        step()  # warm-up
        runs = []
        for _ in range(repeats):
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            step()
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            runs.append(time.perf_counter() - start)
        seconds[name] = min(runs)  # the fastest run: least disturbed by other work on the machine
    return seconds["softmax"], seconds["sgns"]


print(f"One forward + backward pass, batch of 256 pairs, d = {D}, K = {K}\n")
print("     |V|  full softmax   SGNS     ratio")
for _V in (VOCAB_SIZE, 100_000):
    t_soft, t_sgns = time_steps(_V)
    print(f"{_V:>8,}  {t_soft * 1e3:>9.1f} ms  {t_sgns * 1e3:>5.1f} ms  {t_soft / t_sgns:>6.0f}x")

The softmax step costs $O(|V|)$ per pair and grows with the vocabulary; the SGNS step costs $O(K)$ and does not. Our vocabulary is small (12,469 words). Real word2vec vocabularies have hundreds of thousands to millions of words.

## A2 · Training the Embeddings (provided)

The `SkipGram` module is the one in the briefing's PyTorch refresher, with one addition: the initialization of word2vec. $E$ starts with small random values and $U$ starts at zero, so every score is 0 at the first step.

**Predict.** Sketch the loss curve before you run the cell. Where does it start? Does it fall to 0?

In [ ]:
EPOCHS = 3
BATCH = 16_384  # pairs per step
LR = 1e-2


class SkipGram(nn.Module):
    def __init__(self, vocab_size, d):
        super().__init__()
        self.E = nn.Embedding(vocab_size, d)  # center vectors  e_w
        self.U = nn.Embedding(vocab_size, d)  # context vectors u_w
        nn.init.uniform_(self.E.weight, -0.5 / d, 0.5 / d)  # word2vec's initialization
        nn.init.zeros_(self.U.weight)

    def forward(self, center, context, negatives):
        # center, context: (B,)   negatives: (B, K)
        return sgns_loss(self.E(center), self.U(context), self.U(negatives))


torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
sgns = SkipGram(VOCAB_SIZE, D).to(DEVICE)
opt = torch.optim.Adam(sgns.parameters(), lr=LR)
noise = torch.tensor(P_NOISE, dtype=torch.float, device=DEVICE)

losses, started = [], time.time()
for epoch in range(EPOCHS):
    centers, contexts = make_pairs(train_ids, train_doc, word_counts, rng)  # fresh subsample each epoch
    order = rng.permutation(len(centers))
    centers = torch.from_numpy(centers[order]).to(DEVICE)
    contexts = torch.from_numpy(contexts[order]).to(DEVICE)
    for start in range(0, len(centers), BATCH):
        center, context = centers[start : start + BATCH], contexts[start : start + BATCH]
        negatives = torch.multinomial(noise, len(center) * K, replacement=True).view(-1, K)  # (B, K) from P_n
        loss = sgns(center, context, negatives)  # 1-2. batch and forward pass
        opt.zero_grad()  # 3. clear old gradients
        loss.backward()  # 4. backward pass
        opt.step()  # 5. update E and U
        losses.append(loss.item())
    print(f"epoch {epoch + 1}: {len(centers):,} pairs, mean loss {np.mean(losses[-(len(centers) // BATCH):]):.4f}, "
          f"{time.time() - started:.0f} s so far")
SGNS_SECONDS = time.time() - started
E_sgns = sgns.E.weight.detach().cpu()  # (|V|, d): the word vectors. U is discarded.

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(losses, color=NAVY, linewidth=1)
ax.axhline((K + 1) * math.log(2), color=ACCENT, linestyle="--", label="(K + 1) log 2: chance")
ax.set_xlabel("training step")
ax.set_ylabel("SGNS loss")
ax.legend(frameon=False)
plt.show()
print(f"First step {losses[0]:.4f} · last 50 steps {np.mean(losses[-50:]):.4f} · {SGNS_SECONDS:.0f} s on {DEVICE}")
assert abs(losses[0] - (K + 1) * math.log(2)) < 1e-3, "with U = 0 the first loss must be (K + 1) log 2"
assert np.mean(losses[-50:]) < losses[0] - 1.0, "the loss should fall well below its starting value"

**Explain.** The curve starts at $(K + 1)\log 2$, falls quickly, then flattens well above 0. It cannot reach 0: the noise words are drawn at random, so some of them are words that really do occur near the center word, and no setting of the vectors can classify those pairs as fake with certainty. Each epoch also draws fresh pairs and fresh negatives, so this is a loss on data the model has partly not seen. A flat curve here means the vectors have stopped changing much, not that they are perfect.

## Exercise 2 · Nearest Neighbors (7 minutes)

Two words are similar when their vectors point in similar directions. The briefing's @eq-cosine:

$$
\cos(e_a, e_b) = \frac{e_a^\top e_b}{\lVert e_a \rVert \, \lVert e_b \rVert}.
$$

**Predict.** Which words do you expect next to *robot* in embeddings trained on 4,800 arXiv abstracts? Next to *the*?

**Task.** Complete `nearest_neighbors`. The scaffold turns a word query into its vector and builds the set of excluded words; you write the three numbered steps. `torch.topk(x, k)` returns the `k` largest values and their indices, largest first.

<details>
<summary>Hint for TODO 2</summary>

Rank by cosine, not by the raw dot product: divide each dot product by both vector lengths (briefing @eq-cosine), because the vector of a frequent word tends to be long. In the checkpoint, *dog* has the largest dot product with *cat* but points 45 degrees away, while *kitten* points almost the same way. Set the excluded words to $-\infty$ before `torch.topk`, so they can never be picked.

</details>

In [ ]:
# TODO 2: complete this function.
def nearest_neighbors(query, E, k=10, exclude=(), words=None):
    # The k words whose rows of E are most cosine-similar to `query` (briefing @eq-cosine).
    #   query:   a word (str) or a vector of shape (d,)
    #   E:       (|V|, d) embedding matrix; row i is the vector of words[i]
    #   exclude: words never returned; a word query always excludes itself
    #   words:   the list of words, row by row (default: this lab's vocabulary WORDS)
    # Returns a list of k (word, cosine similarity) pairs, most similar first.
    words = WORDS if words is None else words
    index = WORD_TO_ID if words is WORDS else {w: i for i, w in enumerate(words)}
    exclude = set(exclude) | {"<pad>"}
    if isinstance(query, str):
        exclude.add(query)
        query = E[index[query]]
    ...  # 1. the cosine similarity of `query` with every row of E: shape (|V|,)
    ...  # 2. set the similarity of every excluded word to -inf
    ...  # 3. take the k largest and return [(word, similarity), ...]

In [ ]:
#@title Solution 2 — try it yourself first { display-mode: "form" }
@workshop.solution(2)
def nearest_neighbors(query, E, k=10, exclude=(), words=None):
    # The k words whose rows of E are most cosine-similar to `query` (briefing @eq-cosine).
    words = WORDS if words is None else words
    index = WORD_TO_ID if words is WORDS else {w: i for i, w in enumerate(words)}
    exclude = set(exclude) | {"<pad>"}
    if isinstance(query, str):
        exclude.add(query)
        query = E[index[query]]
    sims = (E @ query) / (E.norm(dim=1) * query.norm()).clamp_min(1e-8)  # (|V|,)
    for w in exclude:
        if w in index:
            sims[index[w]] = -float("inf")
    top = torch.topk(sims, k)
    return [(words[i], s) for s, i in zip(top.values.tolist(), top.indices.tolist())]

<details>
<summary>Why this solution works</summary>

`E @ query` gives all $|V|$ dot products $e_v^\top q$ in one matrix-vector product; dividing by the norms turns them into cosines (@eq-cosine). `clamp_min` avoids dividing by zero for an all-zero row. Setting excluded entries to $-\infty$ guarantees `topk` never picks them, whatever the other similarities are.

Cosine, not the raw dot product: the length of a word vector tends to grow with the word's frequency, so ranking by dot product favors frequent words. The checkpoint has a word, *dog*, whose dot product with *cat* is the largest but whose direction is not the closest.

</details>

In [ ]:
# Checkpoint 2
workshop.checkpoint(2)
_words = ["cat", "kitten", "dog", "car", "truck"]
_E = torch.tensor([[1.0, 0.0, 0.0],  # cat
                   [0.9, 0.1, 0.0],  # kitten: almost the same direction as cat
                   [3.0, 3.0, 0.0],  # dog: largest dot product with cat, but 45 degrees away
                   [0.0, 0.0, 1.0],  # car
                   [0.0, 0.1, 0.9]])  # truck
_nn = nearest_neighbors("cat", _E, k=2, words=_words)
assert [w for w, _ in _nn] == ["kitten", "dog"], f"{_nn}: rank by cosine, not by dot product"
assert math.isclose(_nn[0][1], 0.9 / math.sqrt(0.82), rel_tol=1e-5), (
    f"cosine(cat, kitten) = 0.9 / sqrt(0.82) = {0.9 / math.sqrt(0.82):.5f}, got {_nn[0][1]}: "
    "return the cosine, the dot product divided by both norms")
assert math.isclose(_nn[1][1], 1 / math.sqrt(2), rel_tol=1e-5), (
    f"cosine(cat, dog) = 1 / sqrt(2) = {1 / math.sqrt(2):.5f}, got {_nn[1][1]}")
assert all(w != "cat" for w, _ in nearest_neighbors("cat", _E, k=4, words=_words)), "exclude the query word"
assert nearest_neighbors("cat", _E, k=1, exclude={"kitten"}, words=_words)[0][0] == "dog", (
    f"with kitten excluded, the nearest word to cat is dog; got "
    f"{nearest_neighbors('cat', _E, k=1, exclude={'kitten'}, words=_words)}: set every excluded word to -inf")
assert nearest_neighbors(torch.tensor([0.0, 0.0, 2.0]), _E, k=1, words=_words)[0][0] == "car", (
    f"the vector query (0, 0, 2) points exactly at car; got "
    f"{nearest_neighbors(torch.tensor([0.0, 0.0, 2.0]), _E, k=1, words=_words)}")
_sims = [s for _, s in nearest_neighbors("robot", E_sgns, k=10)]
assert len(_sims) == 10 and _sims == sorted(_sims, reverse=True), "return k pairs, most similar first"
print("Checkpoint 2 passed")

**Run.** Neighbors in the embeddings you trained, for a fixed list of probe words. Each probe has a short list of expected neighbors, written down before any embeddings were trained. The **hit rate** is the share of probe words whose 10 nearest neighbors include at least one expected word. It is a rough check, not a benchmark: an expected list can miss good neighbors, and a hit can be luck.

In [ ]:
PROBES = {  # probe word -> neighbors we would accept (fixed before training; not tuned to the results)
    "image": ["images", "pixel", "visual", "video"],
    "robot": ["robots", "robotic", "manipulator", "humanoid"],
    "language": ["languages", "linguistic", "text", "multilingual"],
    "speech": ["audio", "asr", "spoken", "acoustic"],
    "segmentation": ["semantic", "detection", "instance", "masks"],
    "policy": ["reinforcement", "rl", "reward", "policies"],
    "graph": ["graphs", "node", "nodes", "gnn"],
    "grasping": ["grasp", "manipulation", "grasps", "objects"],
    "accuracy": ["performance", "precision", "accuracies"],
    "neural": ["network", "networks", "deep"],
    "camera": ["cameras", "lidar", "depth", "rgb"],
    "theorem": ["bound", "proof", "convergence", "bounds"],
    "navigation": ["planning", "navigate", "path", "obstacle"],
    "video": ["videos", "temporal", "frames", "image"],
    "diffusion": ["generative", "denoising", "gan", "generation"],
    "translation": ["mt", "machine", "translations", "multilingual"],
    "training": ["train", "trained", "pretraining", "finetuning"],
    "llms": ["llm", "gpt", "models"],
}


def neighbor_hit_rate(E, show=True):
    hits = 0
    for probe, expected in PROBES.items():
        if not E[WORD_TO_ID[probe]].any():  # an all-zero row: the word has no vector (stretch section)
            if show:
                print(f"     {probe:<13} (no vector: counted as a miss)")
            continue
        top = [w for w, _ in nearest_neighbors(probe, E, k=10)]
        hit = any(w in top for w in expected)
        hits += hit
        if show:
            print(f"{'hit ' if hit else '    '} {probe:<13} {', '.join(top[:6])}")
    return hits / len(PROBES)


SGNS_HIT_RATE = neighbor_hit_rate(E_sgns)
print(f"\nNeighbor hit rate: {SGNS_HIT_RATE:.2f} ({round(SGNS_HIT_RATE * len(PROBES))} of {len(PROBES)} probe words)")
print("Neighbors of 'the':", [w for w, _ in nearest_neighbors("the", E_sgns, k=6)])
# Measured with the solutions: 0.72 to 0.83 across seeds 0-2 on CPU. A wrong nearest_neighbors scores near 0.
assert SGNS_HIT_RATE >= 0.5, f"hit rate {SGNS_HIT_RATE:.2f}: check Exercise 2 (or the training in A2)"

**Explain.** Most neighbors are topical: words from the same kind of paper, which is what the distributional hypothesis predicts from contexts in abstracts. Look for two other patterns:

- Neighbors that are **related but not similar**: method names, dataset names and benchmark names that appear next to the probe word in abstracts. Cosine similarity measures shared contexts, not shared meaning (misconception 2 in the briefing).
- Rare words among the neighbors. A word seen only a few times has a vector set by a handful of updates, and it can land almost anywhere.

## A3 · Analogies and a Picture of the Space (provided)

The briefing's @eq-analogy answers "$a$ is to $b$ as $a'$ is to ?" with the word nearest to $e_b - e_a + e_{a'}$, excluding the three input words. `analogy` below calls your `nearest_neighbors`. The list was also fixed before training: mostly word forms (plural, *-ing*, comparatives), which a small corpus has the best chance with, and four relations between topics.

**Predict.** What share of the 20 analogies will be answered correctly?

In [ ]:
def analogy(a, b, a2, E):
    # "a is to b as a2 is to ?" by briefing @eq-analogy: the word nearest to e_b - e_a + e_a2, inputs excluded.
    target = E[WORD_TO_ID[b]] - E[WORD_TO_ID[a]] + E[WORD_TO_ID[a2]]  # (d,)
    return nearest_neighbors(target, E, k=1, exclude={a, b, a2})[0][0]


ANALOGIES = [  # (a, b, a2, expected b2): fixed before training
    ("image", "images", "robot", "robots"),
    ("model", "models", "method", "methods"),
    ("network", "networks", "graph", "graphs"),
    ("task", "tasks", "dataset", "datasets"),
    ("video", "videos", "camera", "cameras"),
    ("policy", "policies", "strategy", "strategies"),
    ("train", "training", "learn", "learning"),
    ("plan", "planning", "track", "tracking"),
    ("large", "larger", "small", "smaller"),
    ("high", "higher", "low", "lower"),
    ("fast", "faster", "slow", "slower"),
    ("efficient", "efficiently", "effective", "effectively"),
    ("detect", "detection", "segment", "segmentation"),
    ("generate", "generation", "translate", "translation"),
    ("encoder", "encoding", "decoder", "decoding"),
    ("classification", "classifier", "regression", "regressor"),
    ("vision", "image", "language", "text"),
    ("image", "pixel", "text", "token"),
    ("camera", "image", "microphone", "audio"),
    ("robot", "robotics", "language", "linguistics"),
]


def analogy_accuracy(E, analogies=ANALOGIES, show=True):
    correct = 0
    for a, b, a2, expected in analogies:
        answer = analogy(a, b, a2, E)
        correct += answer == expected
        if show:
            print(f"{'ok' if answer == expected else '  '}  {a} : {b} :: {a2} : {answer:<14} (expected {expected})")
    return correct / len(analogies)


SGNS_ANALOGY = analogy_accuracy(E_sgns)
print(f"\nAnalogy accuracy: {SGNS_ANALOGY:.2f} ({round(SGNS_ANALOGY * len(ANALOGIES))} of {len(ANALOGIES)})")

In [ ]:
# A 2-D picture: PCA of 60 hand-picked words from four groups (fixed before training).
GROUPS = {
    "Vision": ["image", "images", "pixel", "video", "camera", "segmentation", "detection", "visual",
               "depth", "scene", "object", "3d", "rgb", "lidar", "texture"],
    "Language": ["text", "language", "sentence", "word", "words", "translation", "speech", "token",
                 "tokens", "english", "multilingual", "linguistic", "dialogue", "question", "answer"],
    "Robotics": ["robot", "robots", "grasping", "manipulation", "navigation", "locomotion", "legged",
                 "gripper", "arm", "humanoid", "motion", "control", "trajectory", "planning", "drone"],
    "Numbers": ["one", "two", "three", "four", "five", "six", "ten", "first", "second", "third",
                "1", "2", "3", "10", "100"],
}
_rows = [(w, g) for g, ws in GROUPS.items() for w in ws if w in WORD_TO_ID]
_X = E_sgns[[WORD_TO_ID[w] for w, _ in _rows]]
_X = _X / _X.norm(dim=1, keepdim=True)  # unit length: PCA of directions, as cosine sees them
_X = _X - _X.mean(dim=0)
_, _S, _Vt = torch.linalg.svd(_X, full_matrices=False)
_xy = (_X @ _Vt[:2].T).numpy()  # (60, 2) first two principal components
_explained = (_S[:2] ** 2 / (_S**2).sum()).tolist()

fig, ax = plt.subplots(figsize=(7.5, 6))
for color, group in zip(GROUP_COLORS, GROUPS):
    pts = np.array([xy for xy, (_, g) in zip(_xy, _rows) if g == group])
    ax.scatter(pts[:, 0], pts[:, 1], color=color, label=group, s=18)
for (x, y), (w, g) in zip(_xy, _rows):
    ax.annotate(w, (x, y), fontsize=8, xytext=(3, 2), textcoords="offset points",
                color=GROUP_COLORS[list(GROUPS).index(g)])
ax.set_xlabel(f"PC 1 ({_explained[0]:.0%} of variance)")
ax.set_ylabel(f"PC 2 ({_explained[1]:.0%} of variance)")
ax.legend(frameon=False)
ax.set_title("SGNS embeddings, 2-D PCA")
plt.show()

with open("lab02_pca_points.csv", "w", newline="", encoding="utf-8") as f:  # the briefing figure is drawn from this
    _writer = csv.writer(f)
    _writer.writerow(["word", "group", "pc1", "pc2"])
    _writer.writerows([w, g, f"{x:.4f}", f"{y:.4f}"] for (w, g), (x, y) in zip(_rows, _xy))
print(f"{len(_rows)} words; PC 1 and PC 2 explain {_explained[0]:.0%} and {_explained[1]:.0%} of their variance. "
      "Saved to lab02_pca_points.csv")

**Explain.** Write one sentence in the cell below (double-click to edit): pick one analogy that failed and say why it might have failed. Look at what came back instead.

*Your answer:*

Things to notice. In our runs only 4 to 6 of the 20 analogies came out right, as the briefing warned (misconception 3). Most of the successes are word forms, such as *dataset : datasets* and *track : tracking*: a suffix shifts contexts in a regular way that even a small corpus shows many times. Read the failures before you call them wrong. Some are near misses that a person might accept (*method : approach*). Others return a rare word that happens to sit near the target point (a robot model name, a number such as *9x*): with about one million tokens, many vectors rest on a handful of updates, and an offset of two vectors adds the noise of both. The topic relations need words that are rare in abstracts (*microphone*, *linguistics*).

The picture is much cleaner than the analogies. The four groups form four separate clouds, and *motion* and *trajectory* sit between robotics and vision, where papers that use them sit too. Two dimensions keep only about a fifth of the variance of these 60 vectors, so a word that looks misplaced may simply be far away along a direction you cannot see.

# Part B · Embeddings as Features for Classification

Lab 1 classified papers with TF-IDF vectors, one dimension per vocabulary word, and logistic regression. Here each paper becomes the **average of its word embeddings** (briefing @eq-avg), a dense vector of 100 numbers, and a feed-forward network classifies it. The embeddings stay **frozen**: they come from Part A and are not changed by the classifier, so the comparison measures the features.

## Exercise 3 · Averaging Embeddings (6 minutes)

Papers have different lengths, so a batch of them is stored as a padded matrix `token_ids` of shape `(N, T)`, where `T` is the longest paper in the batch and shorter rows are filled with the padding ID 0. A `mask` of the same shape is 1.0 at real tokens and 0.0 at padding. The briefing's @eq-avg, with the mask:

$$
x_n = \frac{\sum_{t} \mathrm{mask}_{n,t}\, e_{\text{token}_{n,t}}}{\sum_t \mathrm{mask}_{n,t}}.
$$

**Predict.** What goes wrong if you average over all `T` positions, padding included? And what does your formula give for a paper with no in-vocabulary tokens?

**Task.** Complete `average_embeddings`. `E[token_ids]` looks up every token at once and has shape `(N, T, d)`.

<details>
<summary>Hint for TODO 3</summary>

Zero the padding positions with the mask before you sum over the time axis, then divide by each paper's number of real tokens, the mask's row sum, not by the padded width `T` (briefing @eq-avg). Dividing by `T` shrinks a short paper's vector by however much padding its batch happened to need. A paper with no real tokens has a count of 0: keep the denominator at least 1, so that 0 / 0 does not give NaN.

</details>

In [ ]:
# TODO 3: complete this function.
def average_embeddings(token_ids, mask, E):
    # Mean of the embeddings of the real tokens of each document (briefing @eq-avg).
    #   token_ids: (N, T) long, padded with 0
    #   mask:      (N, T) float, 1.0 at real tokens and 0.0 at padding
    #   E:         (|V|, d)
    # Returns (N, d). A document with no real tokens gets the zero vector.
    raise NotImplementedError("TODO 3")

In [ ]:
#@title Solution 3 — try it yourself first { display-mode: "form" }
@workshop.solution(3)
def average_embeddings(token_ids, mask, E):
    # Mean of the embeddings of the real tokens of each document (briefing @eq-avg).
    vectors = E[token_ids]  # (N, T, d)
    total = (vectors * mask.unsqueeze(-1)).sum(dim=1)  # (N, d): padding positions contribute 0
    length = mask.sum(dim=1, keepdim=True)  # (N, 1): number of real tokens
    return total / length.clamp_min(1.0)  # an empty document: 0 / 1 = 0, not 0 / 0 = NaN

<details>
<summary>Why this solution works</summary>

Multiplying by the mask zeroes the padding rows before the sum, and dividing by the mask's row sum divides by the true length $T_n$ of each paper, not by the padded width. Averaging over all positions would pull short papers toward the vector of `<pad>` in proportion to how much padding they got, so a paper's features would depend on which other papers shared its batch.

`clamp_min(1.0)` only changes the denominator of a paper with no real tokens, whose numerator is already the zero vector. None of the 7,000 papers is empty after tokenization, but code that silently produces NaN on an edge case breaks later in a way that is hard to trace.

</details>

In [ ]:
# Checkpoint 3
workshop.checkpoint(3)
_g = torch.Generator().manual_seed(0)
_E = torch.randn(10, 4, generator=_g)  # row 0 (the padding ID) is not zero here on purpose
_ids = torch.tensor([[3, 5, 2, 0, 0], [7, 0, 0, 0, 0], [0, 0, 0, 0, 0]])
_mask = (_ids != 0).float()
_out = average_embeddings(_ids, _mask, _E)
assert _out.shape == (3, 4), f"expected shape (N, d) = (3, 4), got {tuple(_out.shape)}"
assert torch.allclose(_out[0], _E[[3, 5, 2]].mean(0), atol=1e-6), (
    f"padding must not change the average: got {_out[0].tolist()}, expected {_E[[3, 5, 2]].mean(0).tolist()}: "
    "zero the padding with the mask and divide by the number of real tokens, not by T")
assert torch.allclose(_out[1], _E[7], atol=1e-6), (
    f"a paper with one real token must get that token's vector {_E[7].tolist()}; got {_out[1].tolist()}")
assert torch.isfinite(_out).all() and torch.equal(_out[2], torch.zeros(4)), (
    f"an all-padding row must give zeros, got {_out[2].tolist()}: keep the denominator at least 1, so 0 / 0 is not NaN")
_unpadded = average_embeddings(torch.tensor([[3, 5, 2]]), torch.ones(1, 3), _E)
assert torch.allclose(_unpadded[0], _out[0], atol=1e-6), "the same document with and without padding must agree"
print("Checkpoint 3 passed")

**Run.** The provided `embed_split` pads each split in batches of 256 papers (so the `(N, T, d)` tensor stays small), calls your function, and returns one 100-dimensional vector per paper. The features are then standardized with the mean and standard deviation of the training split, as the TF-IDF vectors of Lab 1 were normalized; scaling is fitted on train only, like any other fitted parameter.

In [ ]:
def embed_split(ids_list, E, batch=256):
    # Average embeddings for a list of ID arrays, in padded batches. Returns (N, d).
    out = []
    for start in range(0, len(ids_list), batch):
        chunk = ids_list[start : start + batch]
        token_ids = torch.zeros(len(chunk), max(len(ids) for ids in chunk), dtype=torch.long)  # (n, T)
        for row, ids in enumerate(chunk):
            token_ids[row, : len(ids)] = torch.from_numpy(ids)
        out.append(average_embeddings(token_ids, (token_ids != 0).float(), E))
    return torch.cat(out)


def standardized_features(E, ids=None):
    # Averaged-embedding features for train / val / test, standardized with train statistics.
    ids = doc_ids if ids is None else ids
    feats = {split: embed_split(ids[split], E) for split in ("train", "val", "test")}
    mean, std = feats["train"].mean(0), feats["train"].std(0) + 1e-6
    return {split: (x - mean) / std for split, x in feats.items()}


X_sgns = standardized_features(E_sgns)
print({split: tuple(x.shape) for split, x in X_sgns.items()})

## Exercise 4 · A Feed-Forward Classifier (8 minutes)

The briefing's @eq-ffn, for input $x \in \mathbb{R}^d$ and $C = 4$ classes:

$$
h = \mathrm{ReLU}(W_1 x + b_1), \qquad z = W_2 h + b_2, \qquad \mathcal{L} = -\log \mathrm{softmax}(z)_y.
$$

The layers are created in `__init__`: `self.hidden` holds $W_1, b_1$ and `self.out` holds $W_2, b_2$. `forward` returns the **logits** $z$. The softmax and the log are left to `F.cross_entropy`, which computes them together in a numerically stable way.

**Predict.** Without the ReLU, what model would this be?

**Task.** Write `forward`.

In [ ]:
# TODO 4: write forward.
class FeedForwardClassifier(nn.Module):
    # One hidden ReLU layer (briefing @eq-ffn). forward returns logits z, not probabilities.
    def __init__(self, d, hidden, n_classes):
        super().__init__()
        self.hidden = nn.Linear(d, hidden)  # W_1: (hidden, d), b_1: (hidden,)
        self.out = nn.Linear(hidden, n_classes)  # W_2: (C, hidden), b_2: (C,)

    def forward(self, x):
        # x: (B, d) -> logits (B, C)
        raise NotImplementedError("TODO 4")

In [ ]:
#@title Solution 4 — try it yourself first { display-mode: "form" }
@workshop.solution(4)
class FeedForwardClassifier(nn.Module):
    # One hidden ReLU layer (briefing @eq-ffn). forward returns logits z, not probabilities.
    def __init__(self, d, hidden, n_classes):
        super().__init__()
        self.hidden = nn.Linear(d, hidden)  # W_1: (hidden, d), b_1: (hidden,)
        self.out = nn.Linear(hidden, n_classes)  # W_2: (C, hidden), b_2: (C,)

    def forward(self, x):
        h = torch.relu(self.hidden(x))  # (B, hidden)  h = ReLU(W_1 x + b_1)
        return self.out(h)  # (B, C)       z = W_2 h + b_2

<details>
<summary>Why this solution works</summary>

`nn.Linear(d, hidden)` computes $W_1 x + b_1$ for every row of the batch at once; ReLU keeps the positive part; the second layer maps to one logit per class. These are the first two lines of @eq-ffn.

Without the ReLU the two layers compose to $W_2 W_1 x + (W_2 b_1 + b_2)$, a single linear map, and the model is multinomial logistic regression on the averaged embeddings: the same model family as Lab 1, on different features. Returning logits and leaving the softmax to `F.cross_entropy` avoids computing $\log(\exp(\cdot))$ in two separate, less stable steps.

</details>

In [ ]:
# Checkpoint 4
workshop.checkpoint(4)
_m = FeedForwardClassifier(2, 2, 2)
with torch.no_grad():  # set W_1 = W_2 = I and b = 0, so the output shows exactly what forward does
    _m.hidden.weight.copy_(torch.eye(2))
    _m.hidden.bias.zero_()
    _m.out.weight.copy_(torch.eye(2))
    _m.out.bias.zero_()
_z = _m(torch.tensor([[1.0, -2.0]]))
assert torch.allclose(_z, torch.tensor([[1.0, 0.0]])), (
    f"got {_z.tolist()}: expected [[1, 0]]. Is the ReLU applied? Are you returning logits, not a softmax?")
_m = FeedForwardClassifier(D, 256, 4)
assert _m(X_sgns["train"][:32]).shape == (32, 4), "logits must have shape (B, C)"
print("Checkpoint 4 passed")

**Backpropagation, checked.** The first line of the briefing's @eq-backprop says the gradient of the cross-entropy with respect to the logits is $\hat{y} - \mathrm{onehot}(y)$: predicted probabilities minus the target. The cell asks autograd for $\partial \mathcal{L} / \partial z$ on a batch and compares. `F.cross_entropy` averages over the batch, so autograd's answer is divided by $B$.

In [ ]:
torch.manual_seed(SEED)
_m = FeedForwardClassifier(D, 256, 4)
_x, _y = X_sgns["train"][:16], torch.from_numpy(y_train[:16])
_z = _m(_x)  # (16, 4) logits
_z.retain_grad()  # keep the gradient of this intermediate tensor
F.cross_entropy(_z, _y).backward()
_delta_z = (torch.softmax(_z, dim=1) - F.one_hot(_y, 4).float()) / len(_y)  # @eq-backprop, first line, / B
assert torch.allclose(_z.grad, _delta_z.detach(), atol=1e-7)
print("dL/dz from autograd equals (softmax(z) - onehot(y)) / B. First row:")
print("  autograd:", [round(v, 4) for v in _z.grad[0].tolist()])
print("  formula: ", [round(v, 4) for v in _delta_z[0].tolist()])

**Run: train the classifier.** The provided loop trains with Adam on mini-batches for a fixed number of epochs, measures macro-F1 on the validation split after each epoch, and keeps the weights of the best epoch. That choice is made on validation, as Lab 1 chose `C`; the test split is then scored once.

**Predict.** Lab 1's TF-IDF + logistic regression scored about 0.88 test accuracy. Will 100 averaged dimensions do better or worse than 12,469 TF-IDF dimensions? By how much?

In [ ]:
HIDDEN, CLF_EPOCHS, CLF_BATCH, CLF_LR = 256, 30, 128, 1e-3
ACCURACY_FLOOR = 0.85  # the solutions measured 0.864 to 0.872 test accuracy in four CPU runs


def train_classifier(X, seed=SEED):
    # Train a FeedForwardClassifier on X["train"]; keep the epoch with the best val macro-F1.
    torch.manual_seed(seed)
    model = FeedForwardClassifier(X["train"].shape[1], HIDDEN, len(TOPIC_CLASSES))
    opt = torch.optim.Adam(model.parameters(), lr=CLF_LR, weight_decay=1e-4)
    ytr = torch.from_numpy(y_train)
    gen = torch.Generator().manual_seed(seed)
    best_f1, best_epoch, best_state = -1.0, None, None
    for epoch in range(CLF_EPOCHS):
        model.train()
        for idx in torch.randperm(len(ytr), generator=gen).split(CLF_BATCH):
            loss = F.cross_entropy(model(X["train"][idx]), ytr[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_f1 = f1_score(y_val, model(X["val"]).argmax(1).numpy(), average="macro")
        if val_f1 > best_f1:
            best_f1, best_epoch = val_f1, epoch + 1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_epoch


def scores(model, X, split, y):
    with torch.no_grad():
        pred = model(X[split]).argmax(1).numpy()
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"), "pred": pred}


started = time.time()
clf_sgns, best_epoch = train_classifier(X_sgns)
sgns_val = scores(clf_sgns, X_sgns, "val", y_val)
sgns_test = scores(clf_sgns, X_sgns, "test", y_test)  # the test split, once, after the epoch was chosen on val
print(f"Best epoch on val: {best_epoch} of {CLF_EPOCHS}   ({time.time() - started:.0f} s)")
print(f"val  accuracy {sgns_val['accuracy']:.4f} · macro-F1 {sgns_val['macro_f1']:.4f}")
print(f"test accuracy {sgns_test['accuracy']:.4f} · macro-F1 {sgns_test['macro_f1']:.4f}")
assert sgns_test["accuracy"] >= ACCURACY_FLOOR, (
    f"test accuracy {sgns_test['accuracy']:.4f} is below {ACCURACY_FLOOR}: check Exercises 1, 3 and 4")

## The Comparison

The TF-IDF baseline is recomputed here exactly as in Lab 1 (same tokenizer, `min_df=2`, the same `C` grid chosen on validation macro-F1), so this notebook runs without Lab 1's outputs. Both rows are scored on the same 1,600 test papers.

In [ ]:
started = time.time()
counter = CountVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, min_df=2)
counts_train = counter.fit_transform(train_texts)
tfidf = TfidfTransformer().fit(counts_train)
X_tfidf = {"train": tfidf.transform(counts_train), "val": tfidf.transform(counter.transform(val_texts)),
           "test": tfidf.transform(counter.transform(test_texts))}
assert set(counter.vocabulary_) == set(WORDS[1:]), "TF-IDF and the embeddings must share one vocabulary"
lr_val = {}
for C in [0.1, 1.0, 10.0, 100.0]:
    lr = LogisticRegression(C=C, max_iter=1000, random_state=SEED).fit(X_tfidf["train"], y_train)
    lr_val[C] = (f1_score(y_val, lr.predict(X_tfidf["val"]), average="macro"), lr)
BEST_C = max(lr_val, key=lambda C: lr_val[C][0])  # highest val macro-F1; ties go to the smaller C
tfidf_pred = lr_val[BEST_C][1].predict(X_tfidf["test"])
tfidf_test = {"accuracy": accuracy_score(y_test, tfidf_pred), "macro_f1": f1_score(y_test, tfidf_pred, average="macro"),
              "pred": tfidf_pred}
print(f"TF-IDF + logistic regression: C = {BEST_C} chosen on val   ({time.time() - started:.0f} s)\n")

rows = [
    (f"TF-IDF ({X_tfidf['train'].shape[1]:,} dims) + logistic regression", tfidf_test),
    (f"Averaged SGNS ({D} dims, frozen) + feed-forward net", sgns_test),
]
print("arXiv topics, test split (1,600 papers)")
print(f"{'features + model':<52} accuracy  macro-F1   " + "  ".join(f"{n[:6]:>6}" for n in TOPIC_NAMES))
for name, s in rows:
    per_class = f1_score(y_test, s["pred"], average=None)
    print(f"{name:<52} {s['accuracy']:>8.4f}  {s['macro_f1']:>8.4f}   " + "  ".join(f"{v:>6.3f}" for v in per_class))
gap = sgns_test["accuracy"] - tfidf_test["accuracy"]
print(f"\nAveraged embeddings minus TF-IDF: {gap:+.4f} accuracy "
      f"({round(gap * len(y_test)):+d} papers), {sgns_test['macro_f1'] - tfidf_test['macro_f1']:+.4f} macro-F1")
print("(per-class columns are F1)")

**Explain.** Write one sentence below: which row won, by how many papers, and one reason.

*Your answer:*

Things to notice. In our runs TF-IDF won, by about one to two points of accuracy (the measured numbers are in `data/baselines.json`). That is not a failure of the code; it is the empirical answer to the question the briefing left open (misconception 5). Reasons to weigh:

- **Averaging blurs distinctive words.** A topic is often signaled by a few rare, specific words (*lidar*, *parsing*, *convergence*). TF-IDF gives each its own weight, boosted by idf. In the average, they are diluted by the two hundred or so other tokens of the abstract (about 227 on average), punctuation and *the* included, each with equal weight.
- **The labeled data is not scarce.** With 4,800 labeled papers a linear model can learn a weight for most useful words directly. Embeddings help most when labels are few and the embeddings were trained on much more text than the labels cover.
- **Our embeddings saw only about one million tokens.** The stretch section repeats the classifier with GloVe vectors trained on six billion.
- **What the embeddings do give:** 100 numbers instead of 12,469, and a representation in which related words such as *lidar* and *depth* are close, so the classifier can generalize to words it saw rarely with a label.

Your numbers can differ slightly from ours, more on a GPU than on a CPU: the embeddings depend on random pairs and negatives, and the GPU computes in a different order.

## Results Card

In [ ]:
_p, _r, _f, _ = precision_recall_fscore_support(y_test, sgns_test["pred"])
card = {
    "corpus": {"train_tokens": n_train_tokens, "vocabulary": VOCAB_SIZE - 1, "pairs_last_epoch": int(len(centers))},
    "sgns": {"d": D, "window": WINDOW, "K": K, "tau": TAU, "epochs": EPOCHS, "batch": BATCH, "lr": LR,
             "first_loss": round(losses[0], 4), "final_loss_last_50_steps": round(float(np.mean(losses[-50:])), 4),
             "seconds": round(SGNS_SECONDS, 1), "device": str(DEVICE),
             "neighbor_hit_rate": round(SGNS_HIT_RATE, 4), "analogy_accuracy": round(SGNS_ANALOGY, 4)},
    "tfidf_logreg": {"C": BEST_C, "test_accuracy": round(tfidf_test["accuracy"], 4),
                     "test_macro_f1": round(tfidf_test["macro_f1"], 4)},
    "sgns_avg_ffn": {
        "hidden": HIDDEN, "best_epoch": best_epoch,
        "val_accuracy": round(sgns_val["accuracy"], 4), "val_macro_f1": round(sgns_val["macro_f1"], 4),
        "test_accuracy": round(sgns_test["accuracy"], 4), "test_macro_f1": round(sgns_test["macro_f1"], 4),
        "test_per_class": {
            TOPIC_CLASSES[c]: {"precision": round(float(_p[c]), 4), "recall": round(float(_r[c]), 4),
                               "f1": round(float(_f[c]), 4)}
            for c in range(len(TOPIC_CLASSES))
        },
    },
    "versions": {"torch": torch.__version__, "numpy": np.__version__, "scikit-learn": sklearn.__version__},
}
with open("lab02_baselines.json", "w", encoding="utf-8") as f:
    json.dump(card, f, indent=2)

print("RESULTS CARD · Lab 2 · measured in this run")
print("=" * 66)
print(f"SGNS: {EPOCHS} epochs, {card['sgns']['seconds']:.0f} s on {DEVICE}; "
      f"loss {card['sgns']['first_loss']:.3f} -> {card['sgns']['final_loss_last_50_steps']:.3f}")
print(f"Neighbor hit rate {SGNS_HIT_RATE:.2f} · analogy accuracy {SGNS_ANALOGY:.2f}")
for _name, _s in [("TF-IDF + logistic regression", tfidf_test), ("Averaged SGNS + feed-forward", sgns_test)]:
    print(f"{_name:<30} test accuracy {_s['accuracy']:.4f} · macro-F1 {_s['macro_f1']:.4f}")
print("=" * 66)
print(f"Run time so far: {time.time() - LAB_START:.0f} s")

In [ ]:
# Checkpoint: the TF-IDF row reproduces Lab 1's recorded baseline (data/baselines.json, lab01.tfidf_logreg).
workshop.checkpoint(label="baselines")
assert BEST_C == 10.0, BEST_C
assert abs(tfidf_test["accuracy"] - 0.8838) < 0.005, tfidf_test["accuracy"]
assert abs(tfidf_test["macro_f1"] - 0.8841) < 0.005, tfidf_test["macro_f1"]
print("Lab 1 baseline reproduced. This is the end of the core path.")

## Stretch (optional) · Pretrained GloVe Vectors

For those who finish early. Nothing later in the workshop depends on this section.

GloVe (briefing @eq-glove) fits vectors to global co-occurrence counts. We load the 100-dimensional vectors that Stanford trained on Wikipedia 2014 and Gigaword 5, about 6 billion tokens with a 400,000-word vocabulary, and rerun three things with them: the probe neighbors, the analogies, and the Exercise 3 and 4 classifier. We keep only the words of our own vocabulary, so the neighbors and analogies are searched over the same 12,469 words as before.

- **Source.** The file is Stanford's `glove.6B.100d.txt` as converted to word2vec text format and published by the gensim-data project (a GitHub release asset), 134 MB compressed. The download is checked against a SHA-256 hash. Stanford's own download is an 822 MB zip of four files; this one file is what we need.
- **License.** Stanford releases the pretrained vectors under the [Open Data Commons Public Domain Dedication and License](https://opendatacommons.org/licenses/pddl/) (stated in the README of `stanfordnlp/GloVe`).
- **Cost.** Under 2 minutes for the whole section where we ran it (about 4 s to download, a few seconds to read, the rest for the classifier); not yet measured on Colab. If the download fails, the cells below print a message and skip; the rest of the notebook is unaffected.

**Predict.** GloVe saw about 5,000 times more text than our SGNS run, but none of it was chosen for this task. Which of the three checks will improve most? Will any get worse?

In [ ]:
import gzip as _gzip
import hashlib as _hashlib
import os as _os
import urllib.request as _urlrequest
from pathlib import Path as _Path

GLOVE = {  # pinned source; see the markdown above
    "name": "glove-wiki-gigaword-100.gz",
    "url": "https://github.com/RaRe-Technologies/gensim-data/releases/download/"
           "glove-wiki-gigaword-100/glove-wiki-gigaword-100.gz",
    "sha256": "4934d4708bf3d65f28fc984dbbfa18af9cbd2505d166a7258a1a957808559b43",
    "bytes": 134_300_434,
}


def fetch_glove():
    # Download (once) and verify the GloVe file. Returns its path, or None if it cannot be had.
    cache = _Path(_os.environ.get("NLP_LLMS_CACHE", _Path.home() / ".cache" / "nlp-llms"))
    path = cache / GLOVE["name"]

    def ok(p):
        if not p.exists() or p.stat().st_size != GLOVE["bytes"]:
            return False
        digest = _hashlib.sha256()
        with open(p, "rb") as fh:
            for block in iter(lambda: fh.read(1 << 20), b""):
                digest.update(block)
        return digest.hexdigest() == GLOVE["sha256"]

    if ok(path):
        return path
    try:
        cache.mkdir(parents=True, exist_ok=True)
        partial = path.with_suffix(".part")
        with _urlrequest.urlopen(GLOVE["url"], timeout=60) as response, open(partial, "wb") as fh:
            for block in iter(lambda: response.read(1 << 20), b""):
                fh.write(block)
        partial.replace(path)
    except OSError as error:
        print(f"Could not download GloVe: {error}")
        return None
    if not ok(path):
        print("The downloaded GloVe file does not match the expected hash; skipping.")
        return None
    return path


def load_glove(path, words):
    # Rows of GloVe for `words` (missing words stay zero). Returns (matrix (|V|, 100), found mask).
    index = {w: i for i, w in enumerate(words)}
    matrix = torch.zeros(len(words), 100)
    found = torch.zeros(len(words), dtype=torch.bool)
    with _gzip.open(path, "rt", encoding="utf-8") as fh:
        next(fh)  # header line: "400000 100"
        for line in fh:
            word, rest = line.split(" ", 1)
            i = index.get(word)
            if i is not None:
                matrix[i] = torch.tensor([float(v) for v in rest.split()])
                found[i] = True
    return matrix, found


started = time.time()
glove_path = fetch_glove()
if glove_path is None:
    E_glove = None
    print("Skipping the GloVe comparison. Everything above is unaffected.")
else:
    E_glove, glove_found = load_glove(glove_path, WORDS)
    glove_found[0] = False  # <pad>
    _covered = sum(word_counts[i] for i in torch.nonzero(glove_found).flatten().tolist()) / word_counts.sum()
    print(f"GloVe covers {int(glove_found.sum()):,} of {VOCAB_SIZE - 1:,} vocabulary words "
          f"({glove_found.sum().item() / (VOCAB_SIZE - 1):.1%}), {_covered:.1%} of training tokens   "
          f"({time.time() - started:.0f} s)")
    _missing = [w for w in WORDS[1:] if not glove_found[WORD_TO_ID[w]]]
    print("Most frequent words GloVe lacks:", _missing[:15])

In [ ]:
if E_glove is not None:
    # Words GloVe lacks keep an all-zero row: cosine 0 with everything, so in practice never a neighbor.
    print("Neighbors with GloVe, restricted to our vocabulary")
    GLOVE_HIT_RATE = neighbor_hit_rate(E_glove)
    print(f"Neighbor hit rate: GloVe {GLOVE_HIT_RATE:.2f} · our SGNS {SGNS_HIT_RATE:.2f}\n")

    _usable = [an for an in ANALOGIES if all(glove_found[WORD_TO_ID[w]] for w in an)]
    print(f"Analogies with GloVe ({len(_usable)} of {len(ANALOGIES)} have all four words in GloVe)")
    GLOVE_ANALOGY = analogy_accuracy(E_glove, _usable)
    _sgns_same = np.mean([analogy(a, b, a2, E_sgns) == e for a, b, a2, e in _usable])
    print(f"Analogy accuracy on these {len(_usable)}: GloVe {GLOVE_ANALOGY:.2f} · our SGNS {_sgns_same:.2f}")

In [ ]:
if E_glove is not None:
    started = time.time()
    _glove_ids = {split: [ids[glove_found[ids].numpy()] for ids in lst] for split, lst in doc_ids.items()}
    X_glove = standardized_features(E_glove, _glove_ids)  # GloVe-less tokens are dropped, like padding
    clf_glove, _epoch = train_classifier(X_glove)
    glove_test = scores(clf_glove, X_glove, "test", y_test)
    print(f"GloVe classifier: best epoch {_epoch}   ({time.time() - started:.0f} s)\n")
    print(f"{'features + model':<52} accuracy  macro-F1")
    for name, s in rows + [("Averaged GloVe (100 dims, frozen) + feed-forward net", glove_test)]:
        print(f"{name:<52} {s['accuracy']:>8.4f}  {s['macro_f1']:>8.4f}")
    card["glove_avg_ffn"] = {"test_accuracy": round(glove_test["accuracy"], 4),
                             "test_macro_f1": round(glove_test["macro_f1"], 4),
                             "vocabulary_coverage": round(glove_found.sum().item() / (VOCAB_SIZE - 1), 4),
                             "neighbor_hit_rate": round(GLOVE_HIT_RATE, 4),
                             "analogy_accuracy": round(GLOVE_ANALOGY, 4), "analogies_scored": len(_usable)}
    with open("lab02_baselines.json", "w", encoding="utf-8") as f:
        json.dump(card, f, indent=2)

**Explain.** Compare the three checks with your prediction. In our run (CPU; the numbers are in `data/baselines.json`):

- **Analogies improved most:** GloVe answered 11 of the 19 it could score (0.58), our SGNS run 4 of the same 19. Plurals and comparatives are the same in any English text, and 6 billion tokens show each of them many times over. GloVe differs from our run in its objective as well as its data, but controlled comparisons find that the choice between these two objectives matters less than data and hyperparameters (briefing, misconception 3). The amount of data is the likely limit on our analogies.
- **Neighbors got worse:** hit rate 0.61 against our 0.72. GloVe's neighbors are general-English neighbors: *speech* sits next to *address* and *debate*, *diffusion* next to *flux* and *coefficient*, *neural* next to *neurons* and *cortical*. Our SGNS run saw only these abstracts and has the sense these papers use. GloVe also has no vector for *llms*: it was trained on text from 2014.
- **The classifier got much worse:** about 0.80 test accuracy, six points below our own embeddings and eight below TF-IDF, although GloVe covers 99% of the training tokens. The 8% of vocabulary words it lacks are many of the most telling ones (*llms*, *pretrained*, *vlms*, *gnns*), and words with a specific meaning here get their general meaning.

The lesson for the rest of the workshop: more data helps a representation only if it is data about the task. Module 6 returns to this, with a model pretrained on general text and then fine-tuned on these papers.

In [ ]:
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Which checkpoints passed, and on whose code; then a run record for the instructor.
workshop.summary()
workshop.run_record()

<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
---
**Next:** [3 · Sequence Models](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/03-sequence-models.ipynb)

[Workshop site](https://project-delphi.github.io/nlp-llms) · [All notebooks](https://project-delphi.github.io/nlp-llms/notebooks.html) · [Source](https://github.com/project-delphi/nlp-llms)

Genial Labs · text CC BY 4.0, code MIT